# Задача 3. Создание хеш-таблицы

**Условие:** реализовать хеш-таблицу с операциями вставки, поиска по ключу и удаления. Для хранения данных можно использовать только `list`.


## 1. Решение

In [ ]:
class HashTable:

    def __init__(self, capacity=4):
        if type(capacity) is not int or capacity <= 0:
            raise ValueError("Размер таблицы должен быть положительным целым числом")
        self._buckets = [[] for _ in range(capacity)]
        self._size = 0

    @property
    def capacity(self):
        return len(self._buckets)

    def __len__(self):
        return self._size

    def _index(self, key):
        return hash(key) % self.capacity

    def insert(self, key, value):
        bucket = self._buckets[self._index(key)]

        for pair in bucket:
            if pair[0] is key or pair[0] == key:
                pair[1] = value
                return

        if (self._size + 1) / self.capacity > 0.75:
            self._resize()
            bucket = self._buckets[self._index(key)]

        bucket.append([key, value])
        self._size += 1

    def search(self, key):
        bucket = self._buckets[self._index(key)]
        for pair in bucket:
            if pair[0] is key or pair[0] == key:
                return pair[1]
        raise KeyError(key)

    def delete(self, key):
        bucket = self._buckets[self._index(key)]
        for i, pair in enumerate(bucket):
            if pair[0] is key or pair[0] == key:
                del bucket[i]
                self._size -= 1
                return
        raise KeyError(key)

    def _resize(self):
        new_buckets = [[] for _ in range(self.capacity * 2)]
        for bucket in self._buckets:
            for pair in bucket:
                index = hash(pair[0]) % len(new_buckets)
                new_buckets[index].append(pair)
        self._buckets = new_buckets


## 2. Объяснение решения

Храним таблицу в списке `_buckets`. Каждый его элемент — отдельный список, или корзина. Внутри корзины лежат пары `[ключ, значение]`, тоже списки. Словари и множества для хранения данных не используются.

Номер корзины считаем по формуле `hash(key) % capacity`. Встроенная функция `hash` только вычисляет хеш ключа; вставку, поиск, удаление и расширение реализуем самостоятельно.

**Коллизии.** У разных ключей может получиться один номер корзины. Тогда храним несколько пар в одном списке. При поиске сравниваем сами ключи, поэтому одинаковый хеш не делает ключи одинаковыми.

**Вставка.** Сначала проверяем нужную корзину. Если ключ уже есть, заменяем его значение, не увеличивая количество элементов. Если ключ новый, добавляем пару и увеличиваем `_size`.

**Поиск и удаление.** Просматриваем только корзину нужного ключа. При поиске возвращаем значение, при удалении убираем пару и уменьшаем `_size`. Если ключа нет, вызываем `KeyError`. Это позволяет отличать отсутствие ключа от сохранённого значения `None`.

**Расширение.** Следим за отношением количества пар к количеству корзин. Если новая пара поднимет его выше 0.75, удваиваем число корзин. После этого заново вычисляем индексы всех пар: остаток от деления зависит от размера таблицы. Цепочки позволяют хранить данные и при коллизиях, а расширение помогает избежать слишком длинных списков.

Ключи должны быть хешируемыми, например числами или строками. Их хеш и результат сравнения не должны меняться, пока ключ хранится в таблице. Список в качестве ключа использовать нельзя: `hash` вызовет `TypeError`. Значения могут быть любыми.

После каждой операции пара находится в корзине, соответствующей хешу её ключа. Поэтому поиск проверяет именно тот список, где хранится нужное значение. При расширении это правило сохраняется благодаря пересчёту индексов.


## 3. Оценка сложности

Пусть **n** — количество пар, **c** — количество корзин.

| Операция | Среднее время | Худший случай |
|---|---|---|
| Поиск | O(1) | O(n) |
| Удаление | O(1) | O(n) |
| Вставка | O(1) амортизированно | O(n + c), если нужно расширение |
| Расширение | O(n + c) | O(n + c) |

Средние оценки предполагают равномерное распределение хешей. Тогда при заполненности не выше 0.75 цепочки в среднем короткие. Если все ключи попадут в одну корзину, придётся просматривать до n пар.

Одна вставка с расширением дороже обычной: создаём новые корзины и переносим пары. Но размер удваивается, поэтому расширения происходят всё реже, и в среднем на одну вставку приходится O(1) работы при хорошем распределении хешей.

**Память: O(n + c)** — списки корзин и хранимые пары. При расширении временно существуют старый и новый списки корзин, но порядок оценки не меняется. После удаления таблица не сжимается, поэтому её размер зависит и от прежнего количества элементов.

Предполагаем, что вычисление хеша и сравнение ключей занимают O(1). Для длинных строк нужно дополнительно учитывать стоимость этих операций.


## 4. Пошаговая визуализация

Начнём с четырёх корзин и выполним вставки ключей `0`, `4`, `8`, `12`, затем обновление и удаление. У этих небольших целых чисел `hash(key) = key`, поэтому индексы легко посчитать вручную.

В таблице ниже показаны только непустые корзины. Каждая пара хранится как список `[ключ, значение]`.

| Шаг | Операция | Число корзин | Состояние после операции | Что произошло |
|---|---|---:|---|---|
| 1 | `insert(0, "A")` | 4 | №0: `[[0, "A"]]` | `0 % 4 = 0` |
| 2 | `insert(4, "B")` | 4 | №0: `[[0, "A"], [4, "B"]]` | Коллизия: `4 % 4 = 0` |
| 3 | `insert(8, "C")` | 4 | №0: `[[0, "A"], [4, "B"], [8, "C"]]` | Ещё одна коллизия; заполненность 3/4 |
| 4 | `insert(12, "D")` | 8 | №0: `[[0, "A"], [8, "C"]]`; №4: `[[4, "B"], [12, "D"]]` | Перед вставкой расширяем таблицу: 4/4 > 0.75 |
| 5 | `insert(4, "B2")` | 8 | №0: `[[0, "A"], [8, "C"]]`; №4: `[[4, "B2"], [12, "D"]]` | Меняем значение, количество пар остаётся 4 |
| 6 | `delete(0)` | 8 | №0: `[[8, "C"]]`; №4: `[[4, "B2"], [12, "D"]]` | Удаляем только нужную пару, остаётся 3 пары |
| 7 | `search(8)` | 8 | Без изменений | В корзине №0 находим ключ 8 и возвращаем `"C"` |

При расширении пары с ключами `4` и `12` переместились в корзину №4, а `0` и `8` остались в корзине №0. Коллизии сохранились, но цепочки стали короче. Удаление ключа `0` не помешало поиску другого ключа в той же корзине.


In [3]:
example = HashTable(capacity=4)
example.insert(0, "A")
example.insert(4, "B")
example.insert(8, "C")
example.insert(12, "D")
example.insert(4, "B2")
example.delete(0)

print("Поиск ключа 8:", example.search(8))
print("Количество пар:", len(example))
print("Количество корзин:", example.capacity)
print("Корзины:", example._buckets)


Поиск ключа 8: C
Количество пар: 3
Количество корзин: 8
Корзины: [[[8, 'C']], [], [], [], [[4, 'B2'], [12, 'D']], [], [], []]


## 5. Тесты

In [ ]:
import unittest


class CollisionKey:
    def __init__(self, value):
        self.value = value

    def __hash__(self):
        return 1

    def __eq__(self, other):
        return isinstance(other, CollisionKey) and self.value == other.value


class TestHashTable(unittest.TestCase):
    def test_insert_and_search(self):
        table = HashTable()
        for key, value in [["name", "Аня"], [-5, "negative"], [0, [1, 2]]]:
            table.insert(key, value)
            self.assertEqual(table.search(key), value)
        self.assertEqual(len(table), 3)

    def test_update_does_not_resize(self):
        table = HashTable()
        for key in [0, 1, 2]:
            table.insert(key, key)
        table.insert(1, "new")
        self.assertEqual(table.search(1), "new")
        self.assertEqual(len(table), 3)
        self.assertEqual(table.capacity, 4)

    def test_collisions_survive_resize(self):
        table = HashTable()
        for i in range(20):
            table.insert(CollisionKey(i), i * 10)
        self.assertGreater(table.capacity, 4)
        self.assertEqual(len(table), 20)
        for i in range(20):
            self.assertEqual(table.search(CollisionKey(i)), i * 10)
        table.insert(CollisionKey(7), "updated")
        self.assertEqual(table.search(CollisionKey(7)), "updated")
        self.assertEqual(len(table), 20)

    def test_delete_from_chain(self):
        for removed in [0, 4, 8]:
            with self.subTest(removed=removed):
                table = HashTable()
                for key in [0, 4, 8]:
                    table.insert(key, key + 1)
                table.delete(removed)
                self.assertEqual(len(table), 2)
                with self.assertRaises(KeyError):
                    table.search(removed)
                for key in [0, 4, 8]:
                    if key != removed:
                        self.assertEqual(table.search(key), key + 1)

    def test_repeated_resize(self):
        table = HashTable(capacity=1)
        for i in range(100):
            table.insert(i, i * i)
            self.assertLessEqual(len(table) / table.capacity, 0.75)
        for i in range(100):
            self.assertEqual(table.search(i), i * i)
        self.assertEqual(len(table), 100)

    def test_delete_all_and_insert_again(self):
        table = HashTable()
        for i in range(10):
            table.insert(i, i)
        for i in range(10):
            table.delete(i)
        self.assertEqual(len(table), 0)
        self.assertTrue(all(bucket == [] for bucket in table._buckets))
        table.insert(4, "again")
        self.assertEqual(table.search(4), "again")
        self.assertEqual(len(table), 1)

    def test_missing_key(self):
        table = HashTable()
        with self.assertRaises(KeyError):
            table.search("missing")
        with self.assertRaises(KeyError):
            table.delete("missing")
        table.insert(0, "saved")
        with self.assertRaises(KeyError):
            table.delete(4)
        self.assertEqual(table.search(0), "saved")
        self.assertEqual(len(table), 1)

    def test_none_value_and_key(self):
        table = HashTable()
        table.insert("empty", None)
        table.insert(None, "value")
        self.assertIsNone(table.search("empty"))
        self.assertEqual(table.search(None), "value")
        table.delete("empty")
        with self.assertRaises(KeyError):
            table.search("empty")

    def test_unhashable_key(self):
        table = HashTable()
        with self.assertRaises(TypeError):
            table.insert([], "value")
        with self.assertRaises(TypeError):
            table.search([])
        with self.assertRaises(TypeError):
            table.delete([])
        self.assertEqual(len(table), 0)

    def test_invalid_capacity(self):
        for capacity in [0, -1, 2.5, True]:
            with self.subTest(capacity=capacity):
                with self.assertRaises(ValueError):
                    HashTable(capacity)

    def test_storage_uses_lists(self):
        table = HashTable()
        for i in range(10):
            table.insert(i, i)
        self.assertIsInstance(table._buckets, list)
        for bucket in table._buckets:
            self.assertIsInstance(bucket, list)
            for pair in bucket:
                self.assertIsInstance(pair, list)
                self.assertEqual(len(pair), 2)

    def test_visualization_example(self):
        table = HashTable()
        for key, value in [[0, "A"], [4, "B"], [8, "C"]]:
            table.insert(key, value)
        self.assertEqual(table.capacity, 4)
        self.assertEqual(table._buckets[0], [[0, "A"], [4, "B"], [8, "C"]])
        table.insert(12, "D")
        self.assertEqual(table.capacity, 8)
        self.assertEqual(table._buckets[0], [[0, "A"], [8, "C"]])
        self.assertEqual(table._buckets[4], [[4, "B"], [12, "D"]])
        table.insert(4, "B2")
        table.delete(0)
        self.assertEqual(table.search(8), "C")
        self.assertEqual(table._buckets[0], [[8, "C"]])
        self.assertEqual(table._buckets[4], [[4, "B2"], [12, "D"]])
        self.assertEqual(len(table), 3)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestHashTable)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "Есть непройденные тесты"


test_collisions_survive_resize (__main__.TestHashTable.test_collisions_survive_resize) ... ok
test_delete_all_and_insert_again (__main__.TestHashTable.test_delete_all_and_insert_again) ... ok
test_delete_from_chain (__main__.TestHashTable.test_delete_from_chain) ... ok
test_insert_and_search (__main__.TestHashTable.test_insert_and_search) ... ok
test_invalid_capacity (__main__.TestHashTable.test_invalid_capacity) ... ok
test_missing_key (__main__.TestHashTable.test_missing_key) ... ok
test_none_value_and_key (__main__.TestHashTable.test_none_value_and_key) ... ok
test_repeated_resize (__main__.TestHashTable.test_repeated_resize) ... ok
test_storage_uses_lists (__main__.TestHashTable.test_storage_uses_lists) ... ok
test_unhashable_key (__main__.TestHashTable.test_unhashable_key) ... ok
test_update_does_not_resize (__main__.TestHashTable.test_update_does_not_resize) ... ok
test_visualization_example (__main__.TestHashTable.test_visualization_example) ... ok

-----------------------------